# 03. 텍스트 분할 (Text Splitter)

문서를 작은 조각(청크, Chunk)으로 나눕니다.

## 왜 분할이 필요한가?
- LLM의 **컨텍스트 길이 제한**이 있습니다.
- 검색 시 **관련 부분만** 정확하게 찾아야 합니다.
- 청크가 너무 크면 노이즈, 너무 작으면 맥락이 끊깁니다.

## 핵심 파라미터
- `chunk_size`: 청크 최대 길이 (문자 수)
- `chunk_overlap`: 청크 간 겹치는 문자 수 (맥락 유지)

In [1]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/ai_basic.txt", encoding="utf-8")
docs = loader.load()

print(f"원본 문서 길이: {len(docs[0].page_content)} 글자")

원본 문서 길이: 2202 글자


## 1. CharacterTextSplitter

지정한 구분자(separator)를 기준으로 분할합니다.

In [2]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator="\n\n",   # 빈 줄 기준으로 분할
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print(f"청크 수: {len(chunks)}")
print(f"첫 번째 청크 ({len(chunks[0].page_content)} 글자):")
print(chunks[0].page_content)

청크 수: 10
첫 번째 청크 (247 글자):
인공지능(AI)의 기초

인공지능이란 무엇인가?
인공지능(Artificial Intelligence, AI)은 컴퓨터 시스템이 인간의 지능적 행동을 모방하도록 하는 기술입니다. 학습, 추론, 문제 해결, 언어 이해 등의 능력을 포함합니다.

머신러닝이란?
머신러닝(Machine Learning)은 인공지능의 한 분야로, 컴퓨터가 명시적인 프로그래밍 없이 데이터로부터 스스로 학습하는 방법입니다. 지도학습, 비지도학습, 강화학습으로 나뉩니다.


## 2. RecursiveCharacterTextSplitter (권장)

여러 구분자를 순서대로 시도하며 분할합니다.  
`["\n\n", "\n", " ", ""]` 순서로 시도하여 자연스럽게 분할합니다.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print(f"청크 수: {len(chunks)}")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n[청크 {i+1}] ({len(chunk.page_content)} 글자)")
    print(chunk.page_content)

청크 수: 10

[청크 1] (247 글자)
인공지능(AI)의 기초

인공지능이란 무엇인가?
인공지능(Artificial Intelligence, AI)은 컴퓨터 시스템이 인간의 지능적 행동을 모방하도록 하는 기술입니다. 학습, 추론, 문제 해결, 언어 이해 등의 능력을 포함합니다.

머신러닝이란?
머신러닝(Machine Learning)은 인공지능의 한 분야로, 컴퓨터가 명시적인 프로그래밍 없이 데이터로부터 스스로 학습하는 방법입니다. 지도학습, 비지도학습, 강화학습으로 나뉩니다.

[청크 2] (264 글자)
지도학습(Supervised Learning)은 입력과 정답 레이블이 쌍으로 주어진 데이터를 학습합니다. 분류(Classification)와 회귀(Regression) 문제에 사용됩니다. 대표적인 알고리즘으로는 선형 회귀, 결정 트리, 랜덤 포레스트, SVM 등이 있습니다.

비지도학습(Unsupervised Learning)은 정답 레이블 없이 데이터의 패턴을 스스로 찾습니다. 클러스터링(K-means, DBSCAN)과 차원 축소(PCA, t-SNE)가 대표적입니다.

[청크 3] (191 글자)
강화학습(Reinforcement Learning)은 에이전트가 환경과 상호작용하면서 보상을 최대화하는 방향으로 학습합니다. 게임 AI, 로봇 제어 등에 활용됩니다.

딥러닝이란?
딥러닝(Deep Learning)은 여러 층의 인공 신경망을 사용하는 머신러닝 기법입니다. 이미지 인식, 자연어 처리, 음성 인식 등에서 뛰어난 성능을 보입니다.


## 3. chunk_size와 chunk_overlap 영향 비교

In [4]:
sample_text = docs[0].page_content

for chunk_size, overlap in [(200, 0), (200, 50), (500, 100)]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    chunks = splitter.split_text(sample_text)
    print(f"chunk_size={chunk_size}, overlap={overlap} → 청크 수: {len(chunks)}")

chunk_size=200, overlap=0 → 청크 수: 16
chunk_size=200, overlap=50 → 청크 수: 16
chunk_size=500, overlap=100 → 청크 수: 5


## 4. overlap 효과 확인

인접한 청크 사이에 겹치는 부분이 있어 맥락이 유지됩니다.

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print("[청크 1 끝부분]")
print(repr(chunks[0].page_content[-60:]))

print("\n[청크 2 앞부분]")
print(repr(chunks[1].page_content[:60]))

print("\n→ 위 두 부분이 겹쳐서 맥락이 이어집니다.")

[청크 1 끝부분]
'의 지능적 행동을 모방하도록 하는 기술입니다. 학습, 추론, 문제 해결, 언어 이해 등의 능력을 포함합니다.'

[청크 2 앞부분]
'머신러닝이란?\n머신러닝(Machine Learning)은 인공지능의 한 분야로, 컴퓨터가 명시적인 프로그래밍'

→ 위 두 부분이 겹쳐서 맥락이 이어집니다.


## 5. 메타데이터 유지 확인

In [6]:
# split_documents는 원본 메타데이터를 각 청크에 유지
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

for chunk in chunks[:3]:
    print("메타데이터:", chunk.metadata)

메타데이터: {'source': 'data/ai_basic.txt'}
메타데이터: {'source': 'data/ai_basic.txt'}
메타데이터: {'source': 'data/ai_basic.txt'}


## 정리

| 분할기 | 특징 |
|---|---|
| CharacterTextSplitter | 단일 구분자 기준 분할 |
| **RecursiveCharacterTextSplitter** | 여러 구분자 순차 시도, **일반적으로 권장** |

- `chunk_size`: 너무 크면 검색 정확도↓, 너무 작으면 맥락 손실
- `chunk_overlap`: 청크 간 맥락 유지, 일반적으로 chunk_size의 10~20%

다음 단계: **임베딩(Embedding)** →